# VoiceFL — Dataset Portrait & MAML Engine Explained

**What this notebook does:**
1. Visualises the final processed dataset (everything after the pipeline ran)
2. Shows the exact shape of every tensor that flows through the system
3. Explains the MAML engine step-by-step like you've never seen backpropagation before

**Pre-requisite:** `data/pii_masking.py → features.py → partition.py` must have run.  
`data/partition_manifest.json` must exist.

In [ ]:
import json
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap

# Go up one level so imports from data/ work
sys.path.insert(0, os.path.abspath('..'))

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'text.color':       '#eee',
    'grid.color':       '#2a2a3e',
    'grid.linewidth':   0.5,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'font.family':      'monospace',
})

ACCENT  = '#7c83fd'   # indigo
ACCENT2 = '#f72585'   # pink
ACCENT3 = '#4cc9f0'   # cyan
ACCENT4 = '#f9c74f'   # amber
GREEN   = '#06d6a0'

print('Imports OK')

---
## Part 1 — Dataset Portrait
### 1.1 Load the manifest

In [ ]:
manifest_path = os.path.join('..', 'data', 'partition_manifest.json')
with open(manifest_path) as f:
    manifest = json.load(f)

nodes = manifest['nodes']
n_nodes = manifest['n_nodes']

node_ids       = [n['node_id'] for n in nodes]
clip_counts    = np.array([n['clip_count'] for n in nodes])
durations_min  = np.array([n['total_duration_s'] / 60 for n in nodes])
mean_dur       = np.array([n['mean_clip_duration_s'] for n in nodes])
std_dur        = np.array([n['std_clip_duration_s'] for n in nodes])
min_dur        = np.array([n['min_duration_s'] for n in nodes])
max_dur        = np.array([n['max_duration_s'] for n in nodes])
vocab_richness = np.array([n['vocabulary_richness'] for n in nodes])
short_ids      = [f"N{int(n['node_id'].split('_')[1]):02d}" for n in nodes]

print(f"Nodes:          {n_nodes}")
print(f"Total clips:    {manifest['total_clips']:,}")
print(f"Total audio:    {manifest['total_duration_hours']:.2f} hours")
print(f"PII status:     {manifest['pii_status']}")
print(f"MAML ready:     {manifest['maml_ready']}")
print(f"Feature format: {manifest['feature_config']['format']}")

### 1.2 Clips per node & total audio duration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('VoiceFL-MAML — Dataset Overview (20 FL Nodes)', fontsize=15, y=1.02)

# --- Left: clips per node ---
ax = axes[0]
colors = [ACCENT if c >= 100 else ACCENT4 for c in clip_counts]
bars = ax.bar(short_ids, clip_counts, color=colors, width=0.7, zorder=2)
ax.axhline(clip_counts.mean(), color=ACCENT2, linewidth=1.5, linestyle='--', label=f'mean = {clip_counts.mean():.0f}')
ax.axhline(50, color='#666', linewidth=1, linestyle=':', label='min threshold (50)')
ax.set_title('Clips per Node')
ax.set_xlabel('Node')
ax.set_ylabel('Clip count')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
ax.grid(axis='y', zorder=1)
ax.set_ylim(0, clip_counts.max() * 1.15)
# annotate min & max
mi, ma = np.argmin(clip_counts), np.argmax(clip_counts)
ax.annotate(f'{clip_counts[mi]}', (short_ids[mi], clip_counts[mi] + 2), ha='center', fontsize=9, color=ACCENT4)
ax.annotate(f'{clip_counts[ma]}', (short_ids[ma], clip_counts[ma] + 2), ha='center', fontsize=9, color=ACCENT)

# --- Right: audio duration per node ---
ax = axes[1]
bars2 = ax.bar(short_ids, durations_min, color=ACCENT3, width=0.7, zorder=2)
ax.axhline(durations_min.mean(), color=ACCENT2, linewidth=1.5, linestyle='--', label=f'mean = {durations_min.mean():.1f} min')
ax.set_title('Total Audio per Node (minutes)')
ax.set_xlabel('Node')
ax.set_ylabel('Minutes')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
ax.grid(axis='y', zorder=1)
ax.set_ylim(0, durations_min.max() * 1.15)

plt.tight_layout()
plt.show()
print(f"\nSmallest node: N{np.argmin(clip_counts)+1:02d} — {clip_counts.min()} clips / {durations_min.min():.1f} min")
print(f"Largest node:  N{np.argmax(clip_counts)+1:02d} — {clip_counts.max()} clips / {durations_min.max():.1f} min")

### 1.3 Non-IID heterogeneity — why this dataset is interesting for FL

**Non-IID** = the data is NOT identically distributed across nodes.  
Each node is a different speaker. Different voices, different reading styles, different vocabulary.  

We measure heterogeneity two ways:
- **Vocabulary richness** — type-token ratio (unique words / total words). Higher = more diverse vocabulary.
- **Clip duration std** — standard deviation of clip lengths. Higher = more variable speech patterns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Non-IID Heterogeneity Across Nodes', fontsize=15, y=1.02)

# --- Left: vocab richness ---
ax = axes[0]
sorted_idx = np.argsort(vocab_richness)
sorted_ids = [short_ids[i] for i in sorted_idx]
sorted_vr  = vocab_richness[sorted_idx]
cmap_vals  = (sorted_vr - sorted_vr.min()) / (sorted_vr.max() - sorted_vr.min())
bar_colors = plt.cm.plasma(cmap_vals * 0.7 + 0.15)
ax.barh(sorted_ids, sorted_vr, color=bar_colors, height=0.7, zorder=2)
ax.axvline(vocab_richness.mean(), color=ACCENT2, linewidth=1.5, linestyle='--', label=f'mean = {vocab_richness.mean():.3f}')
ax.set_title('Vocabulary Richness (type-token ratio)\nHigher = more unique words = more diverse speaker')
ax.set_xlabel('TTR')
ax.legend(fontsize=9)
ax.grid(axis='x', zorder=1)

# --- Right: clip duration std ---
ax = axes[1]
sorted_idx2 = np.argsort(std_dur)
sorted_ids2  = [short_ids[i] for i in sorted_idx2]
sorted_std   = std_dur[sorted_idx2]
cmap_vals2   = (sorted_std - sorted_std.min()) / (sorted_std.max() - sorted_std.min())
bar_colors2  = plt.cm.viridis(cmap_vals2 * 0.7 + 0.15)
ax.barh(sorted_ids2, sorted_std, color=bar_colors2, height=0.7, zorder=2)
ax.axvline(std_dur.mean(), color=ACCENT2, linewidth=1.5, linestyle='--', label=f'mean = {std_dur.mean():.2f}s')
ax.set_title('Clip Duration Std Dev (seconds)\nHigher = more variable speech lengths')
ax.set_xlabel('Std (s)')
ax.legend(fontsize=9)
ax.grid(axis='x', zorder=1)

plt.tight_layout()
plt.show()

# Correlation
corr = np.corrcoef(vocab_richness, std_dur)[0, 1]
print(f"Pearson correlation (vocab richness vs duration std): {corr:.3f}")
print(f"\nInterpretation: speakers with higher vocab richness {'tend to also have' if corr > 0.3 else "don't necessarily have"} more variable clip lengths")

### 1.4 Clip duration distribution — the shape the model sees

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Clip Duration Distribution Across All Nodes', fontsize=15, y=1.02)

# --- Left: box plot per node ---
ax = axes[0]
box_data = []
for n in nodes:
    # Reconstruct approximate distribution from stats (we only have summary stats in manifest)
    m, s = n['mean_clip_duration_s'], n['std_clip_duration_s']
    lo, hi = n['min_duration_s'], n['max_duration_s']
    # Clip a normal distribution to [min, max]
    samples = np.clip(np.random.normal(m, s, 200), lo, hi)
    box_data.append(samples)

bp = ax.boxplot(box_data, labels=short_ids, patch_artist=True, notch=False,
                medianprops=dict(color=ACCENT2, linewidth=2),
                whiskerprops=dict(color='#666'),
                capprops=dict(color='#666'),
                flierprops=dict(marker='.', color='#555', markersize=2))
for patch in bp['boxes']:
    patch.set_facecolor(ACCENT + '33')  # semi-transparent
    patch.set_edgecolor(ACCENT)

ax.set_title('Clip Duration per Node (approximate from summary stats)')
ax.set_xlabel('Node')
ax.set_ylabel('Duration (seconds)')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y')

# --- Right: mean ± std scatter ---
ax = axes[1]
x = np.arange(n_nodes)
ax.errorbar(x, mean_dur, yerr=std_dur, fmt='o', color=ACCENT3,
            ecolor=ACCENT3 + '88', capsize=4, capthick=1.5, linewidth=0, markersize=6)
ax.fill_between(x, mean_dur - std_dur, mean_dur + std_dur, alpha=0.15, color=ACCENT3)
ax.set_xticks(x)
ax.set_xticklabels(short_ids, rotation=45, fontsize=8)
ax.set_title('Mean ± 1 Std Clip Duration per Node')
ax.set_xlabel('Node')
ax.set_ylabel('Duration (seconds)')
ax.grid()

plt.tight_layout()
plt.show()

print(f"Overall mean clip duration: {mean_dur.mean():.2f}s")
print(f"Node with shortest clips:   N{np.argmin(mean_dur)+1:02d} ({mean_dur.min():.2f}s avg)")
print(f"Node with longest clips:    N{np.argmax(mean_dur)+1:02d} ({mean_dur.max():.2f}s avg)")

### 1.5 Full dataset heatmap — all stats at once

In [ ]:
# Build a (n_nodes × 4) matrix — normalise each column to [0,1]
raw = np.column_stack([clip_counts, durations_min, vocab_richness, std_dur])
col_min = raw.min(axis=0)
col_max = raw.max(axis=0)
norm = (raw - col_min) / (col_max - col_min + 1e-8)

col_labels = ['Clip\ncount', 'Audio\n(min)', 'Vocab\nrichness', 'Duration\nstd']

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(norm, aspect='auto', cmap='plasma', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Normalised value (0=min, 1=max)', shrink=0.6)

ax.set_xticks(range(4))
ax.set_xticklabels(col_labels, fontsize=11)
ax.set_yticks(range(n_nodes))
ax.set_yticklabels(short_ids, fontsize=9)
ax.set_title('Node Characteristics Heatmap\n(brighter = higher relative value)', fontsize=13)

# Annotate raw values
raw_fmt = [
    [f'{clip_counts[r]}'     for r in range(n_nodes)],
    [f'{durations_min[r]:.0f}m' for r in range(n_nodes)],
    [f'{vocab_richness[r]:.2f}' for r in range(n_nodes)],
    [f'{std_dur[r]:.1f}s'    for r in range(n_nodes)],
]
for col_i in range(4):
    for row_i in range(n_nodes):
        ax.text(col_i, row_i, raw_fmt[col_i][row_i],
                ha='center', va='center', fontsize=8,
                color='white' if norm[row_i, col_i] < 0.6 else 'black')

plt.tight_layout()
plt.show()

---
## Part 2 — Data Shape: What the Model Actually Sees

Every tensor in this pipeline has a specific shape. Let's make it concrete.

### 2.1 Raw waveform — the input to `features.py`

In [ ]:
import torch

# Try to load one real features.pt — show actual data if it exists
node_dir = os.path.join('..', 'data', 'nodes', 'node_001')
features_path = os.path.join(node_dir, 'features.pt')

if os.path.exists(features_path):
    clips = torch.load(features_path, weights_only=True)
    use_real = True
    print(f'Loaded real features.pt from node_001')
    print(f'Number of clips: {len(clips)}')
    print(f'dtype:           {clips[0].dtype}')
    print(f'Shape of clip 0: {clips[0].shape}  ({clips[0].shape[0]/16000:.2f}s @ 16kHz)')
    print(f'Shape of clip 5: {clips[5].shape}  ({clips[5].shape[0]/16000:.2f}s @ 16kHz)')
    print(f'Value range:     [{clips[0].min():.4f}, {clips[0].max():.4f}]')
    print(f'\nAll clips have DIFFERENT lengths (variable-length audio):')
    for i in [0, 1, 2, 3, 4]:
        print(f'  clip[{i}]: {clips[i].shape[0]:>7} samples = {clips[i].shape[0]/16000:.2f}s')
else:
    use_real = False
    print('features.pt not found — run: python data/features.py')
    print('Using synthetic example for illustration.')
    # Create synthetic clips with realistic shapes
    t_samples = [int(d * 16000) for d in [14.53, 9.2, 16.0, 3.5, 12.8]]
    clips = [torch.randn(t).clamp(-1, 1) for t in t_samples]
    print(f'\nSynthetic example shapes:')
    for i, c in enumerate(clips):
        print(f'  clip[{i}]: {c.shape[0]:>7} samples = {c.shape[0]/16000:.2f}s')

### 2.2 Shape diagram — from single clip to model input

In [ ]:
# Visualise the first real (or synthetic) clip waveform
clip = clips[0].numpy()
time_s = np.linspace(0, len(clip)/16000, len(clip))

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)
fig.patch.set_facecolor('#0f0f0f')
fig.suptitle('Data Shape: From Raw Audio to Batched Tensor', fontsize=14, y=0.98)

# ── Panel 1: Raw waveform (single clip) ──────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(time_s, clip, color=ACCENT3, linewidth=0.5, alpha=0.9)
ax1.fill_between(time_s, clip, alpha=0.2, color=ACCENT3)
ax1.set_title(f'Step 1: features.pt — 1 clip   shape: ({len(clip)},)   dtype: float32   range: [-1, 1]')
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('Amplitude')
ax1.set_xlim(0, time_s[-1])
ax1.grid()
ax1.text(0.02, 0.9, f'{len(clip)} samples @ 16 kHz = {len(clip)/16000:.2f}s',
         transform=ax1.transAxes, fontsize=10, color=ACCENT4,
         bbox=dict(boxstyle='round', facecolor='#1a1a2e', edgecolor=ACCENT4, alpha=0.8))

# ── Panel 2: Variable-length clips before padding ─────────────────────
ax2 = fig.add_subplot(gs[1, 0])
clips_sample = clips[:8] if len(clips) >= 8 else clips
lengths = [c.shape[0] for c in clips_sample]
y_pos = np.arange(len(lengths))
colors_bar = plt.cm.cool(np.linspace(0.2, 0.9, len(lengths)))
for i, (l, col) in enumerate(zip(lengths, colors_bar)):
    ax2.barh(i, l, color=col, height=0.6, edgecolor='none')
    ax2.text(l + 200, i, f'{l/16000:.2f}s', va='center', fontsize=8, color='#aaa')
ax2.set_yticks(y_pos)
ax2.set_yticklabels([f'clip {i}' for i in range(len(lengths))], fontsize=9)
ax2.set_xlabel('Samples')
ax2.set_title('Step 2: K clips BEFORE padding\n(variable lengths — all 16 kHz, float32)')
ax2.grid(axis='x')
ax2.set_xlim(0, max(lengths) * 1.2)

# ── Panel 3: After Wav2Vec2Processor padding → (K, T_max) ────────────
ax3 = fig.add_subplot(gs[1, 1])
K = len(lengths)
T_max = max(lengths)
# Create visual: filled = real samples, empty = padding
grid_data = np.zeros((K, T_max // 1000))  # compress x-axis
for i, l in enumerate(lengths):
    grid_data[i, :l//1000] = 1.0
    grid_data[i, l//1000:] = -0.3  # padding

ax3.imshow(grid_data, aspect='auto', cmap='RdYlBu', vmin=-0.5, vmax=1.1, interpolation='nearest')
ax3.set_title(f'Step 3: AFTER Wav2Vec2Processor padding\nshape: ({K}, {T_max})  =  (K, T_max)')
ax3.set_xlabel(f'Sample index (×1000)  →  T_max = {T_max:,} samples')
ax3.set_ylabel('Clip index')
# Add legend
real_patch   = mpatches.Patch(color='#4575b4', label='Real audio')
pad_patch    = mpatches.Patch(color='#d73027', label='Zero padding')
ax3.legend(handles=[real_patch, pad_patch], loc='lower right', fontsize=9)

# ── Panel 4: Full MAML task shape ────────────────────────────────────
ax4 = fig.add_subplot(gs[2, :])
ax4.set_xlim(0, 10)
ax4.set_ylim(0, 4)
ax4.axis('off')
ax4.set_title('Step 4: A complete MAML task — all 4 tensors', pad=10)

task_tensors = [
    ('support_audio',  f'(K, T_max)  =  (8, {T_max:,})', 'float32', ACCENT),
    ('support_labels', '(K, N_tok)   =  (8, ~60)',        'int64 / -100', ACCENT2),
    ('query_audio',    f'(Q, T_max)  =  (8, {T_max:,})', 'float32', ACCENT3),
    ('query_labels',   '(Q, N_tok)   =  (8, ~60)',        'int64 / -100', ACCENT4),
]

for i, (name, shape, dtype, color) in enumerate(task_tensors):
    x0 = (i % 2) * 5 + 0.3
    y0 = 2.3 - (i // 2) * 1.8
    rect = FancyBboxPatch((x0, y0), 4.4, 1.4, boxstyle='round,pad=0.1',
                          edgecolor=color, facecolor=color + '22', linewidth=2)
    ax4.add_patch(rect)
    ax4.text(x0 + 2.2, y0 + 0.9, name, ha='center', va='center',
             fontsize=12, fontweight='bold', color=color)
    ax4.text(x0 + 2.2, y0 + 0.5, f'shape: {shape}', ha='center', va='center',
             fontsize=9, color='#ccc')
    ax4.text(x0 + 2.2, y0 + 0.15, f'dtype: {dtype}', ha='center', va='center',
             fontsize=8, color='#888')

plt.show()

print('\nShape summary:')
print(f'  features.pt per node:  List[Tensor(T_samples,)]  — variable T, float32, 16kHz, [-1,1]')
print(f'  support_audio:         ({K}, {T_max})          — K=8, padded to longest in batch')
print(f'  support_labels:        ({K}, N_tok)             — tokenized text, -100 = padding')
print(f'  query_audio:           ({K}, {T_max})           — same shape as support')
print(f'  query_labels:          ({K}, N_tok)             — same shape as support_labels')

### 2.3 Model output shape

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_facecolor('#0f0f0f')
fig.patch.set_facecolor('#0f0f0f')
ax.set_title('Wav2Vec2 Forward Pass — Tensor Shapes Through the Model', fontsize=13, pad=15)

stages = [
    ('INPUT\naudio\n(K, T_max)\nfloat32', 0.5, ACCENT3),
    ('Feature\nEncoder\n(CNN layers)\n↓', 2.5, '#888'),
    ('ENCODER\nhidden\n(K, T_frames, 768)\nT_frames ≈ T_max/320', 4.5, ACCENT),
    ('lm_head\n(linear)\n768→32\n↓', 8.0, '#888'),
    ('LOGITS\n(K, T_frames, 32)\nfloat32\n32 = vocab size', 10.0, ACCENT2),
    ('CTC Loss\n↓\nscalar', 12.5, ACCENT4),
]

for label, x, color in stages:
    if '↓' in label and len(label) < 20:
        ax.text(x, 2.5, label, ha='center', va='center', fontsize=12, color=color)
    else:
        rect = FancyBboxPatch((x - 0.9, 1.2), 1.8, 2.6, boxstyle='round,pad=0.08',
                              edgecolor=color, facecolor=color + '22', linewidth=2)
        ax.add_patch(rect)
        ax.text(x, 2.5, label, ha='center', va='center', fontsize=9, color='#eee')

# arrows
arrow_x = [(1.4, 1.7), (3.4, 3.7), (5.4, 7.1), (8.9, 9.1), (11.0, 11.7)]
for x1, x2 in arrow_x:
    ax.annotate('', xy=(x2, 2.5), xytext=(x1, 2.5),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))

# Key annotation
ax.text(7.0, 0.4,
        'T_frames = T_max / 320   (Wav2Vec2 CNN stride)   |   vocab: 32 chars (A-Z + space + blank + 2 special)',
        ha='center', fontsize=9, color='#888',
        bbox=dict(boxstyle='round', facecolor='#1a1a2e', edgecolor='#444'))

plt.tight_layout()
plt.show()

# Show ANIL split visually
print('ANIL split summary:')
print(f'  Encoder (wav2vec2.*):  ~94,000,000 params  → FL aggregated (transmitted)')
print(f'  lm_head:               ~    24,576 params  → stays local (never leaves device)')
print(f'  Ratio:                 encoder is {94000000 // 24576}× larger than lm_head')
print(f'\n  Inner loop adapts:     lm_head only    (fast — ~25K params, 3 SGD steps)')
print(f'  Meta-gradient flows:   query loss → all params, but only encoder grad transmitted')

---
## Part 3 — MAML Engine: Explained Like You're 5

---

### The Big Idea (no code, no math)

Imagine you're running a school for 20 students.  
Each student speaks differently — different accent, different vocabulary, different rhythm.

**Normal AI training (centralised):**  
You collect all 20 students' voices and train one model that's OK for everyone but great for nobody.  
Also, you have all their raw voice recordings — a privacy nightmare.

**VoiceFL-MAML:**  
You don't collect anyone's voice. Instead you ask:
> *"Can I build a model so smart that any new student can make it great for them with just 8 examples and 3 training steps?"*

That's MAML. It doesn't learn to be good at speech recognition.  
**It learns to be good at learning speech recognition quickly.**

---

### The Two Loops

MAML has two nested loops. Think of them as two teachers:

| | Inner loop | Outer loop |
|---|---|---|
| **Who runs it** | Each student (FL node) | The school (FL server) |
| **What it does** | Adapts model to this speaker in K steps | Updates the global model to be easier to adapt |
| **What it changes** | `lm_head` params (25K params) | `encoder` params (94M params) |
| **How many steps** | K = 3 gradient steps | 1 step per FL round |
| **What data** | Support set (8 clips) | Query set (8 different clips) |
| **Travels over network?** | No | Yes (meta-gradient, DP-sanitised) |

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')
fig.patch.set_facecolor('#0a0a1a')
ax.set_facecolor('#0a0a1a')
ax.set_title('FOMAML Step-by-Step (one task, one node)', fontsize=15, color='white', pad=20)

def box(ax, x, y, w, h, label, sublabel='', color=ACCENT, fontsize=10):
    rect = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle='round,pad=0.1',
                          edgecolor=color, facecolor=color + '25', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y + (0.15 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize, color='white', fontweight='bold')
    if sublabel:
        ax.text(x, y - 0.3, sublabel, ha='center', va='center', fontsize=8, color='#aaa')

def arrow(ax, x1, y1, x2, y2, label='', color='#666'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8,
                                connectionstyle='arc3,rad=0'))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.1, my, label, fontsize=8, color=color, va='center')

# ── Step labels on left ──────────────────────────────────────────────
steps = [
    (0.6, 7.8, '①'),
    (0.6, 6.0, '②'),
    (0.6, 4.0, '③'),
    (0.6, 2.2, '④'),
    (0.6, 0.7, '⑤'),
]
for x, y, t in steps:
    ax.text(x, y, t, fontsize=18, color=ACCENT4, ha='center', va='center')

# ── Step 1: Start with θ (global model from server) ──────────────────
box(ax, 4.0, 7.8, 3.5, 0.9, 'Global model  θ*', 'encoder + lm_head', ACCENT)
box(ax, 8.5, 7.8, 3.0, 0.9, 'deepcopy → save  θ*', 'we will restore this later', '#888')
arrow(ax, 5.75, 7.8, 7.0, 7.8, '', '#555')
ax.text(1.5, 7.8, 'Server sends\nθ* to node', ha='center', fontsize=9, color='#999')

# ── Step 2: Inner loop ──────────────────────────────────────────────
box(ax, 2.5, 6.0, 2.8, 1.0, 'Support set\n(8 clips)', 'Speaker voice samples', ACCENT3, fontsize=9)
# 3 inner steps
for i, xi in enumerate([5.5, 7.5, 9.5]):
    box(ax, xi, 6.0, 1.5, 0.8, f'SGD step {i+1}\n(lm_head only)', '', ACCENT2, fontsize=8)
    if i == 0:
        arrow(ax, 3.9, 6.0, 4.75, 6.0, '', ACCENT3)
    else:
        arrow(ax, xi - 0.75 - 0.75, 6.0, xi - 0.75, 6.0, '', ACCENT2)

box(ax, 12.0, 6.0, 2.2, 0.8, 'Adapted  θ\'\n(lm_head updated)', '', GREEN, fontsize=9)
arrow(ax, 10.25, 6.0, 10.9, 6.0, '', ACCENT2)

ax.text(1.2, 5.5, 'Inner loop\nK=3 steps\nlm_head only', ha='center', fontsize=9, color=ACCENT2)

# ── Step 3: Eval on query ────────────────────────────────────────────
box(ax, 3.5, 4.0, 2.8, 1.0, 'Query set\n(8 different clips)', 'NOT the same clips!', ACCENT4, fontsize=9)
box(ax, 8.0, 4.0, 3.5, 1.0, 'forward pass with  θ\'\n→ CTC loss (scalar)', 'how good is the adapted model?', GREEN, fontsize=9)
box(ax, 12.5, 4.0, 2.5, 0.8, 'query_loss\n(scalar)', '', ACCENT2, fontsize=9)
arrow(ax, 4.9, 4.0, 6.25, 4.0, '', ACCENT4)
arrow(ax, 9.75, 4.0, 11.25, 4.0, '', GREEN)
ax.text(1.2, 4.0, 'Evaluate\nadapted model\non unseen data', ha='center', fontsize=9, color=ACCENT4)

# ── Step 4: Compute meta-gradient ───────────────────────────────────
box(ax, 4.5, 2.2, 3.8, 1.0, 'autograd.grad(\n  query_loss, θ*.params,\n  create_graph=False)', 'FOMAML: no Hessian', ACCENT, fontsize=8)
box(ax, 10.0, 2.2, 3.5, 1.0, 'meta_gradient\n∂L_query / ∂θ*', 'list of tensors, same shapes as encoder', ACCENT, fontsize=9)
arrow(ax, 6.4, 2.2, 8.25, 2.2, '', ACCENT)
arrow(ax, 12.5, 3.6, 12.5, 2.7, '', ACCENT2)  # from query_loss down
ax.text(1.2, 2.2, 'Compute\ngradient\n(first-order)', ha='center', fontsize=9, color=ACCENT)

# ── Step 5: Restore θ, return grad ──────────────────────────────────
box(ax, 3.5, 0.7, 3.5, 0.85, 'Restore θ*  (load saved copy)', 'model goes back to its original state', '#888', fontsize=8)
box(ax, 9.0, 0.7, 3.5, 0.85, 'Return meta_gradient  →  DP clip+noise  →  FL server', 'encoder grads only (I3)', ACCENT4, fontsize=8)
arrow(ax, 5.25, 0.7, 7.25, 0.7, '', '#555')
ax.text(1.2, 0.7, 'Restore &\nreturn', ha='center', fontsize=9, color='#999')

plt.tight_layout()
plt.show()

### 3.2 The FOMAML Step-by-Step in Plain English

Here's every line of `_fomaml()` in `maml/engine.py`, translated to plain English:

---

**① Save a snapshot**
```python
init_state = copy.deepcopy(self.model.model.state_dict())
```
> Take a photo of the model right now. We're about to mess with it temporarily, and we need to be able to undo that.

---

**② Inner loop — adapt for this speaker**
```python
inner_opt = torch.optim.SGD(self.model.get_inner_loop_params(), lr=1e-4)
for _ in range(k=3):
    out = self.model(support_audio, support_labels)   # forward pass
    out.loss.backward()                               # how wrong were we?
    inner_opt.step()                                  # nudge lm_head toward correct
```
> Show the model 8 example clips from this speaker. Let it adjust its **output layer** (lm_head) 3 times to get better at this specific voice. The big encoder is NOT touched here — only the tiny 25K-param output layer adapts.

---

**③ Evaluate on the query set**
```python
query_out = self.model(query_audio, query_labels)     # forward pass at θ'
```
> Now test the adapted model on 8 **different** clips (the query set — never seen before).  
> If the adaptation was useful, query loss should be low. If adaptation overfit to the support, query loss stays high.  
> This is the true measure of "did the adaptation generalise?"

---

**④ Compute the meta-gradient**
```python
meta_grads = torch.autograd.grad(
    query_out.loss,
    self.model.model.parameters(),
    create_graph=False     # ← this is the FOMAML vs full-MAML distinction
)
```
> This is the key question: **"How should we change θ* (the global starting point) so that next time, after 3 adaptation steps, the model is even better at the query set?"**
>
> `create_graph=False` = FOMAML. We compute the gradient **at θ'** (the adapted weights), but we pretend it came from θ* directly. No Hessian. This is 2-3× cheaper and works almost as well in practice.
>
> `create_graph=True` = Full MAML. We differentiate **through** the inner loop steps themselves. This gives the exact second-order gradient but is broken with CTC loss (see Known Issues).

---

**⑤ Restore the model, return the gradient**
```python
self.model.model.load_state_dict(init_state)   # undo the adaptation
return list(meta_grads), query_loss_val
```
> Restore the model to its original state (θ* not θ'). The adaptation was **temporary** — it was only there so we could measure how good it was.  
> Return the meta-gradient. The FL client will then DP-sanitise it (clip + Gaussian noise) and send it to the server.

---

**On the server (Per-FedAvg):**
```
θ*  ←  θ* − β · weighted_avg(meta_grads from all nodes)
```
> Nudge the global starting point θ* in the direction that made all nodes adapt better. This is gradient descent on the server — NOT the usual FedAvg weight averaging.

---

### The Loop Repeats

After 50 FL rounds, θ* has been nudged 50 times by 20 speakers' meta-gradients.  
It's no longer a generic speech model. It's a model designed to **adapt quickly** to any voice.  
A new speaker can send 8 clips, run 3 gradient steps locally, and get a personalised model.

No raw audio ever leaves the device. Only gradients. Only those gradients are DP-noised.

In [ ]:
# Visual: 50 rounds of Per-FedAvg — what converges
np.random.seed(42)
rounds = np.arange(1, 51)

# Simulate WER curves (realistic shapes, not real data)
def simulate_wer(rounds, start, end, noise_scale=0.015, decay=0.08):
    trend = end + (start - end) * np.exp(-decay * rounds)
    return trend + np.random.normal(0, noise_scale, len(rounds))

wer_k0  = simulate_wer(rounds, 0.95, 0.72, 0.02, 0.06)  # no adaptation
wer_k1  = simulate_wer(rounds, 0.88, 0.58, 0.018, 0.07) # 1-shot
wer_k3  = simulate_wer(rounds, 0.80, 0.42, 0.015, 0.08) # 3-shot (our default)
wer_k5  = simulate_wer(rounds, 0.77, 0.38, 0.013, 0.09) # 5-shot
wer_k10 = simulate_wer(rounds, 0.74, 0.35, 0.012, 0.09) # 10-shot

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('What MAML Training Achieves (simulated curves — not real run)', fontsize=13)

# --- Left: WER over FL rounds for different k ---
ax = axes[0]
for wer, k, color in [
    (wer_k0,  0,  '#888'),
    (wer_k1,  1,  ACCENT4),
    (wer_k3,  3,  ACCENT),   # our default
    (wer_k5,  5,  ACCENT3),
    (wer_k10, 10, GREEN),
]:
    lw = 2.5 if k == 3 else 1.5
    ls = '-' if k == 3 else '--'
    ax.plot(rounds, wer * 100, color=color, linewidth=lw, linestyle=ls,
            label=f'k={k}{" ← default" if k==3 else ""}')

ax.set_xlabel('FL Round')
ax.set_ylabel('WER (%)')
ax.set_title('WER vs FL Round at different adaptation steps k\n(k=0 = zero-shot; k=3 = 3 inner loop steps)')
ax.legend(fontsize=9)
ax.grid()
ax.set_ylim(0, 105)
# Gate annotation
ax.axhline(wer_k0[-1] * 100, color='#666', linewidth=0.8, linestyle=':')
ax.text(48, wer_k0[-1] * 100 + 1.5, 'k=0 baseline', ha='right', fontsize=8, color='#888')

# --- Right: Adaptation gain (k=3 vs k=0) ---
ax = axes[1]
gain = (wer_k0 - wer_k3) * 100
ax.fill_between(rounds, 0, gain, alpha=0.3, color=GREEN)
ax.plot(rounds, gain, color=GREEN, linewidth=2)
ax.axhline(0, color='#666', linewidth=0.8)
# Gate line
gate_round = 25
ax.axvline(gate_round, color=ACCENT2, linewidth=1.5, linestyle='--', label='approximate gate round')
ax.set_xlabel('FL Round')
ax.set_ylabel('WER reduction (pp)')
ax.set_title('Adaptation Gain: WER(k=0) − WER(k=3)\nGate = gain > 0 on ≥12/20 nodes')
ax.legend(fontsize=9)
ax.grid()
ax.text(gate_round + 0.5, gain.max() * 0.9, 'gate typically\npasses here', fontsize=8, color=ACCENT2)

plt.tight_layout()
plt.show()

print('Reading the graphs:')
print('  Left:  WER drops over rounds for all k — the global model θ* is getting better at being adapted')
print('  Right: Adaptation GAIN grows over rounds — MAML is learning to learn')
print('         When this stays positive: k=3 is better than k=0 → MAML works')
print(f'  Gate:  WER(k=3) < WER(k=0) on ≥12/20 nodes → proceed to federated run')

### 3.3 FOMAML vs Full MAML vs Reptile — the actual difference

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
fig.patch.set_facecolor('#0a0a1a')
ax.set_facecolor('#0a0a1a')
ax.set_title('Three MAML Modes — What Makes Them Different', fontsize=14, color='white', pad=15)

modes = [
    {
        'name': 'FOMAML',
        'subtitle': 'First-Order MAML  ← dev default',
        'color': ACCENT,
        'x': 2.3,
        'steps': [
            ('① Save θ*', 0.4),
            ('② K SGD steps on lm_head\n   (support set)', 0.8),
            ('③ Forward pass at θ\'\n   (query set) → loss', 0.9),
            ('④ grad(loss, θ*.params,\n   create_graph=False)', 0.9),
            ('⑤ Restore θ*, return grad', 0.4),
        ],
        'note': 'Approximation: acts as if gradient at\nθ\' came directly from θ*. No Hessian.\nFaster, works great in practice.',
    },
    {
        'name': 'Full MAML',
        'subtitle': 'Second-Order  ← BROKEN with CTC',
        'color': ACCENT2,
        'x': 7.0,
        'steps': [
            ('① higher.innerloop_ctx(\n   track_higher_grads=True)', 0.8),
            ('② K differentiable steps\n   (builds computation graph)', 0.9),
            ('③ Forward pass at θ\'\n   (query set) → loss', 0.9),
            ('④ grad flows THROUGH\n   inner loop → Hessian included', 0.9),
            ('⑤ Exact Per-FedAvg gradient', 0.4),
        ],
        'note': '⚠ PyTorch has no derivative for\naten::_ctc_loss_backward.\nAlways fails with Wav2Vec2 CTC.',
    },
    {
        'name': 'Reptile',
        'subtitle': 'Reptile (Nichol et al.)  ← fallback',
        'color': ACCENT3,
        'x': 11.7,
        'steps': [
            ('① Save θ* params', 0.4),
            ('② K SGD steps on FULL\n   model (support set)', 0.9),
            ('③ meta_grad = (θ* − θ\') / k\n   no query set needed for grad', 0.9),
            ('④ Restore θ*, return meta_grad', 0.4),
            ('   (query set → loss only,\n    no gradient computation)', 0.8),
        ],
        'note': 'Simplest of the three.\nNo second-order, no backward\nthrough inner loop.\nSlightly less accurate but robust.',
    },
]

for mode in modes:
    x = mode['x']
    color = mode['color']
    # Header
    rect = FancyBboxPatch((x - 2.0, 5.8), 4.0, 1.0, boxstyle='round,pad=0.1',
                          edgecolor=color, facecolor=color + '33', linewidth=2.5)
    ax.add_patch(rect)
    ax.text(x, 6.4, mode['name'], ha='center', fontsize=13, fontweight='bold', color=color)
    ax.text(x, 6.05, mode['subtitle'], ha='center', fontsize=8, color='#bbb')
    
    # Steps
    y = 5.2
    for step_text, step_h in mode['steps']:
        y -= step_h * 0.5
        ax.text(x, y, step_text, ha='center', va='center',
                fontsize=8, color='#ddd',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#1a1a2e', edgecolor='#333', alpha=0.8))
        y -= step_h * 0.6
        if y > 1.2:
            ax.annotate('', xy=(x, y + 0.05), xytext=(x, y + 0.2),
                        arrowprops=dict(arrowstyle='->', color='#555', lw=1))
    
    # Note
    ax.text(x, 0.5, mode['note'], ha='center', va='center',
            fontsize=8, color='#999', style='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#12121e', edgecolor='#333', alpha=0.9))

plt.tight_layout()
plt.show()

### 3.4 The key formula, demystified

**Per-FedAvg server update:**

```
θ*  ←  θ*  −  β  ·  (1/N) Σᵢ meta_gradᵢ
         ↑       ↑        ↑
       global   outer    average of
        model   lr β=2e-4 all nodes'
        params            meta-gradients
```

**What this is NOT:**  
Standard FedAvg averages the model **weights** themselves: `θ* = (1/N) Σ θᵢ`.  
That makes no sense for MAML — clients don't fully train a model, they compute a gradient signal.

**What this IS:**  
Gradient descent on the server. The server uses the aggregated meta-gradients as a single gradient step.  
θ* moves in the direction that makes the average node adapt better.

---

### 3.5 Summary: One FL round, end to end

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16)
ax.set_ylim(0, 6)
ax.axis('off')
fig.patch.set_facecolor('#0a0a1a')
ax.set_facecolor('#0a0a1a')
ax.set_title('One Complete FL Round — Server + 2 of 20 Nodes', fontsize=13, color='white', pad=15)

# Server
rect = FancyBboxPatch((6.5, 4.0), 3.0, 1.5, boxstyle='round,pad=0.15',
                      edgecolor=ACCENT4, facecolor=ACCENT4+'22', linewidth=2.5)
ax.add_patch(rect)
ax.text(8.0, 4.9, 'FL SERVER', ha='center', fontsize=12, fontweight='bold', color=ACCENT4)
ax.text(8.0, 4.35, 'θ* (global encoder)\nPerFedAvgStrategy', ha='center', fontsize=8, color='#bbb')

# Node A and Node B
for i, (xn, node_n) in enumerate([(2.5, 'Node A'), (13.5, 'Node B')]):
    rect = FancyBboxPatch((xn - 2.0, 1.5), 4.0, 2.2, boxstyle='round,pad=0.15',
                          edgecolor=ACCENT, facecolor=ACCENT+'18', linewidth=2)
    ax.add_patch(rect)
    ax.text(xn, 3.3, node_n, ha='center', fontsize=11, fontweight='bold', color=ACCENT)
    ax.text(xn, 2.9, 'MAMLClient', ha='center', fontsize=8, color='#888')
    lines = [
        '① set_parameters(θ*)',
        '② check ε budget',
        '③ sample_task() → sup + qry',
        '④ compute_meta_gradient()',
        '⑤ apply_dp() → sanitised grad',
    ]
    for j, line in enumerate(lines):
        ax.text(xn, 2.55 - j * 0.3, line, ha='center', fontsize=7.5, color='#ccc')

# Arrows: server → nodes (θ*)
ax.annotate('', xy=(4.5, 3.5), xytext=(6.5, 4.3),
            arrowprops=dict(arrowstyle='->', color=ACCENT4, lw=1.8))
ax.text(5.1, 4.1, 'send θ*', fontsize=9, color=ACCENT4, rotation=-20)

ax.annotate('', xy=(11.5, 3.5), xytext=(9.5, 4.3),
            arrowprops=dict(arrowstyle='->', color=ACCENT4, lw=1.8))
ax.text(10.1, 4.1, 'send θ*', fontsize=9, color=ACCENT4, rotation=20)

# Arrows: nodes → server (gradients)
ax.annotate('', xy=(6.5, 4.0), xytext=(4.5, 2.8),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.8))
ax.text(4.8, 3.2, 'meta_grad\n(DP noised)', fontsize=8, color=GREEN, rotation=30)

ax.annotate('', xy=(9.5, 4.0), xytext=(11.5, 2.8),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.8))
ax.text(10.2, 3.2, 'meta_grad\n(DP noised)', fontsize=8, color=GREEN, rotation=-30)

# Server aggregation
rect2 = FancyBboxPatch((5.5, 0.2), 5.0, 1.0, boxstyle='round,pad=0.1',
                       edgecolor=ACCENT2, facecolor=ACCENT2+'22', linewidth=2)
ax.add_patch(rect2)
ax.text(8.0, 0.75, 'BAE screening → weighted_avg(grads) → θ* ← θ* − β · avg_grad', 
        ha='center', fontsize=9, color='#eee')
ax.text(8.0, 0.38, 'PerFedAvgStrategy.aggregate_fit()   →   new θ*   →   next round', 
        ha='center', fontsize=8, color='#888')

ax.annotate('', xy=(8.0, 1.2), xytext=(8.0, 4.0),
            arrowprops=dict(arrowstyle='->', color=ACCENT2, lw=1.5, linestyle='dashed'))

plt.tight_layout()
plt.show()

print('One FL round checklist:')
print('  Server sends θ* to 2 nodes (clients_per_round=2 in dev config)')
print('  Each node runs FOMAML: adapt lm_head → evaluate on query → compute meta-grad')
print('  Each node DP-sanitises the encoder meta-grad (L2 clip + Gaussian noise)')
print('  Server runs BAE screening on received gradients')
print('  Server averages gradients and takes one gradient-descent step on θ*')
print('  θ* is now slightly better at being adapted — repeat 49 more times')

---
## Summary

| Concept | One-line version |
|---------|------------------|
| **LibriSpeech node** | 56–137 clips, ~12–25 min audio, 1D float32 waveforms at 16kHz |
| **feature shape** | `List[Tensor(T_samples,)]` — variable T, float32, [-1,1] |
| **MAML task** | `(support_audio, support_labels, query_audio, query_labels)` — 4 tensors, shape `(8, T_max)` each |
| **ANIL split** | lm_head (25K params) adapts locally; encoder (94M params) is FL-aggregated |
| **FOMAML** | Adapt lm_head on support → eval query → gradient at adapted params → restore original |
| **Meta-gradient** | "How should θ* change so future adaptation on this speaker works better?" |
| **Per-FedAvg** | Server does gradient descent (`θ* ← θ* − β·avg_grad`), NOT weight averaging |
| **DP** | Clip meta-grad L2 norm + add Gaussian noise before transmission |
| **BAE** | 4-layer anomaly score on meta-grads — quarantine or exclude byzantine nodes |
| **Success gate** | WER(k=3) < WER(k=0) on ≥12/20 nodes — confirms MAML actually helps |